In [ ]:
#%pip install requests beautifulsoup4 trafilatura pypdf pandas tqdm nltk
%pip install transformers torch accelerate sentencepiece

In [ ]:
import re
import json
import requests
import pandas as pd

from bs4 import BeautifulSoup
from pypdf import PdfReader
from tqdm import tqdm

import trafilatura
import nltk

nltk.download("punkt")

In [ ]:
def extract_from_url(url):
    """
    Extract clean article-like text from a Privacy Policy URL.
    """

    downloaded = trafilatura.fetch_url(url)

    if downloaded is None:
        raise ValueError("Could not download URL")

    text = trafilatura.extract(
        downloaded,
        include_comments=False,
        include_tables=True
    )

    if text is None:
        raise ValueError("No text extracted")

    return text

In [ ]:
def extract_from_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page in reader.pages:
        txt = page.extract_text()

        if txt:
            pages.append(txt)

    return "\n".join(pages)

In [ ]:
import re

def clean_text(text):

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    text = re.sub(
        r'Cookie Preferences',
        '',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'Skip to main content',
        '',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'Contents',
        '',
        text,
        flags=re.IGNORECASE
    )

    return text.strip()

In [ ]:
def remove_repeated_lines(text, threshold=3):

    lines = [line.strip() for line in text.split("\n")]

    counts = {}

    for line in lines:
        counts[line] = counts.get(line, 0) + 1

    cleaned = [
        line
        for line in lines
        if counts[line] < threshold
    ]

    return "\n".join(cleaned)

In [ ]:
PRIVACY_KEYWORDS = [
    "personal information",
    "privacy policy",
    "information we collect",
    "cookies",
    "data retention",
    "third-party",
    "data sharing",
    "your rights",
    "gdpr",
    "ccpa",
    "children",
    "security",
    "contact us"
]


def privacy_relevance_score(text):

    text_lower = text.lower()

    score = 0

    for kw in PRIVACY_KEYWORDS:
        score += text_lower.count(kw)

    return score

In [ ]:
import re

def remove_table_of_contents(text):

    toc_patterns = [
        r"table of contents",
        r"contents",
        r"index"
    ]

    text_lower = text.lower()

    for pattern in toc_patterns:

        match = re.search(pattern, text_lower)

        if match:

            start = match.start()

            # look ahead ~3000 chars
            window = text[start:start+3000]

            lines = window.split("\n")

            toc_end = 0

            consecutive_non_toc = 0

            for i, line in enumerate(lines):

                line = line.strip()

                if (
                    re.search(r"\.{2,}", line)
                    or re.search(r"\d+$", line)
                    or len(line.split()) < 10
                ):
                    consecutive_non_toc = 0

                else:
                    consecutive_non_toc += 1

                if consecutive_non_toc >= 5:
                    toc_end = i
                    break

            cleaned_window = "\n".join(lines[toc_end:])

            text = text[:start] + cleaned_window

            break

    return text

In [ ]:
NOISE_PATTERNS = [
    "accept all cookies",
    "cookie settings",
    "back to top",
    "print this page",
    "previous",
    "next"
]

def remove_noise(text):

    lines = text.split("\n")

    cleaned = []

    for line in lines:

        lower = line.lower()

        if any(pattern in lower for pattern in NOISE_PATTERNS):
            continue

        cleaned.append(line)

    return "\n".join(cleaned)

In [ ]:
def chunk_text(
    text,
    chunk_size=500,
    overlap=100
):
    text = remove_noise(text)
    words = text.split()

    
    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
def is_toc_chunk(chunk):

    lines = chunk.split("\n")

    short_lines = 0

    dotted_lines = 0

    for line in lines:

        line = line.strip()

        if len(line.split()) < 8:
            short_lines += 1

        if re.search(r"\.{2,}", line):
            dotted_lines += 1

    if short_lines > len(lines) * 0.7:
        return True

    if dotted_lines > 3:
        return True

    return False

In [ ]:
def process_document(source, source_type="url"):

    if source_type == "url":
        raw_text = extract_from_url(source)

    else:
        raw_text = extract_from_pdf(source)

    text = clean_text(raw_text)

    text = remove_repeated_lines(text)

    text = remove_table_of_contents(text)

    chunks = chunk_text(text)

    rows = []

    for idx, chunk in enumerate(chunks):

        if is_toc_chunk(chunk):
            continue

        rows.append({
            "chunk_id": idx,
            "text": chunk
        })

    return pd.DataFrame(rows)

In [ ]:
url = "https://policies.google.com/privacy?hl=en-US"

df = process_document(
    url,
    source_type="url"
)

df.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np

In [ ]:
vectorizer = CountVectorizer(
    stop_words="english",
    lowercase=True
)

bow_matrix = vectorizer.fit_transform(df["text"])

In [ ]:
def bow_retrieve(
    query,
    top_k=5
):

    query_vec = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vec,
        bow_matrix
    ).flatten()

    top_idx = np.argsort(scores)[::-1][:top_k]

    return df.iloc[top_idx][["chunk_id","text"]]

In [ ]:
results = bow_retrieve(
    "How long do you retain user data?"
)

results

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

KeyboardInterrupt: 

In [ ]:
def build_context(retrieved_df):

    context = "\n\n".join(
        retrieved_df["text"].tolist()
    )

    return context

In [ ]:
retrieved = tfidf_retrieve(
    "How long do you retain user data?",
    top_k=5
)

context = build_context(retrieved)

In [ ]:
def create_prompt(
    context,
    question
):

    prompt = f"""
You are an expert Privacy Policy analyst.

Answer ONLY using the provided policy excerpts.

If the information is not explicitly stated,
reply:

Information not found in the policy.

Privacy Policy Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [ ]:
def generate_answer(
    prompt,
    max_new_tokens=256
):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.0,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
query = "How long do you retain personal information?"

retrieved = bow_retrieve(
    query,
    top_k=5
)

context = build_context(retrieved)

prompt = create_prompt(
    context,
    query
)

answer = generate_answer(prompt)

print(answer)

In [ ]:
def rag_answer(
    question,
    retrieval_fn,
    top_k=5
):

    retrieved = retrieval_fn(
        question,
        top_k=top_k
    )

    context = build_context(
        retrieved
    )

    prompt = create_prompt(
        context,
        question
    )

    answer = generate_answer(
        prompt
    )

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved,
        "context": context
    }

In [ ]:
result = rag_answer(
    question="Can users request deletion of their personal information?",
    retrieval_fn=bow_retrieve
)

print(result["answer"])